In [2]:
import os
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SparkExample") \
    .config("spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
        "ru.yandex.clickhouse:clickhouse-jdbc:0.3.2,"
        "org.postgresql:postgresql:42.5.0,"
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0",
        ) \
    .getOrCreate()


hadoop_conf = spark._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", os.getenv("MINIO_ROOT_USER"))
hadoop_conf.set("fs.s3a.secret.key", os.getenv("MINIO_ROOT_PASSWORD"))
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")
hadoop_conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hadoop_conf.set("fs.s3a.path.style.access", "true")


:: loading settings :: url = jar:file:/usr/local/lib/python3.11/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
ru.yandex.clickhouse#clickhouse-jdbc added as a dependency
org.postgresql#postgresql added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-5220172b-d73b-4950-a0f7-4539a7faf8eb;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found ru.yandex.clickhouse#clickhouse-jdbc;0.3.2 in central
	found com.clickhouse#clickhouse-http-client;0.3.2 in central
	found com.clickhouse#clickhouse-client;0.3.2 in central
	found org.lz4#lz4-java;1.8.0 in central
	found com.google.code.gson#gson;2.8.8 in central
	found org.apache.httpcomponents#httpclient;4.5

In [3]:
# ⬇️ Параметры подключения к PostgreSQL 

def pg_table_load(spark, table_name):

	jdbc_url = "jdbc:postgresql://postgres_source:5432/source"
	db_user = os.getenv("POSTGRES_USER")
	db_password = os.getenv("POSTGRES_PASSWORD")

	return (
		spark
		.read 
		.format("jdbc")
		.option("url", jdbc_url)
		.option("user", db_user)
		.option("password", db_password)
		.option("dbtable", table_name)
		.option("fetchsize", 1000)
		.option("driver", "org.postgresql.Driver")
		.load()
	)

shop_timezone_df = pg_table_load(spark, "public.shop_timezone")
shops_df = pg_table_load(spark, "public.shops")

In [4]:
import pyspark.sql.functions as F


# shops_df.show()
# shop_timezone_df.show()

st_timezone = (
	shop_timezone_df
	.join(shops_df, shop_timezone_df.plant==shops_df.st_id, "inner")
	.select("st_id", "shop_name", "time_zone")
    .groupBy("st_id", "shop_name")
    .agg(F.max("time_zone").alias("tz_code"))
	.withColumn(
		"tz_code",
		F.when(F.col("tz_code") == "", "3")
		.otherwise(F.substring(F.col("tz_code"), 5, 1))
	)
)

st_timezone.show()


+-----+-----------+-------+
|st_id|  shop_name|tz_code|
+-----+-----------+-------+
|  842|      Lenta|      7|
|  843|     Magnit|      4|
|  844|       Spar|      3|
|  845|Pyaterochka|      5|
|  847|      Diksi|      3|
|  848|      Lenta|      8|
+-----+-----------+-------+

